# Day 43 — Solutions

## K-Means clustering

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | Running the loop | 1 – 3 |
| **B** | Choosing k | 4 – 6 |
| **C** | Breaking it | 7 – 10 |
| **D** | Using it | 11 – 12 |

Run the setup cell first. Every question is self-contained after that.

> This is the solutions file. Each question shows working code and then the reasoning.
> Read the reasoning — most of the marks are there.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import load_iris, load_wine, make_blobs, make_moons

TEAL, CORAL, VIOLET, INK = "#1F9E86", "#E05C45", "#8268B5", "#131F33"
COLS = [TEAL, CORAL, VIOLET, "#3E7CB1", "#F0A530", "#7B8AA1"]

def show(X, labels=None, centres=None, title="", ax=None):
    ax = ax or plt.gca()
    c = "#9AA8BE" if labels is None else [COLS[l % 6] for l in labels]
    ax.scatter(X[:, 0], X[:, 1], c=c, s=20, edgecolors="white", linewidths=.4, zorder=3)
    if centres is not None:
        ax.scatter(centres[:, 0], centres[:, 1], marker="X", s=280, c=INK,
                   edgecolors="white", linewidths=1.6, zorder=6)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

# ---- Part A and B use this: 260 gym members, nobody has labelled them -----
#      visits per month  x  average minutes per visit
def load_gym(seed=11):
    r = np.random.default_rng(seed)
    a = np.c_[r.normal(19, 3.0, 95), r.normal(38, 7, 95)]     # frequent, short sessions
    b = np.c_[r.normal(6.5, 2.0, 85), r.normal(95, 14, 85)]   # rare, long sessions
    c = np.c_[r.normal(7.0, 2.1, 80), r.normal(35, 7, 80)]    # rare, short sessions
    X = np.vstack([a, b, c])
    truth = np.r_[np.zeros(95), np.ones(85), np.full(80, 2)].astype(int)
    X[:, 0] = X[:, 0].clip(1, None); X[:, 1] = X[:, 1].clip(10, None)
    i = r.permutation(len(X))
    return X[i], truth[i]

GYM, GYM_TRUTH = load_gym()
print("gym members:", len(GYM))
print(f"visits   {GYM[:,0].min():5.1f} to {GYM[:,0].max():5.1f}   spread {GYM[:,0].std():6.2f}")
print(f"minutes  {GYM[:,1].min():5.1f} to {GYM[:,1].max():5.1f}   spread {GYM[:,1].std():6.2f}")

---
# Part A · Running the loop

## Question 1

Write the two steps of the algorithm yourself.

- `assign(Z, centres)` — return the index of the nearest centre for every point
- `move(Z, labels, k)` — return each centre moved to the mean of its own points
- `inertia_of(Z, labels, centres)` — total **squared** distance from each point to its own centre

Scale the gym data first, drop 3 centres at random, then run **one** assign and **one**
move, printing the inertia after each.

In [ ]:
Z = StandardScaler().fit_transform(GYM)

def assign(Z, centres):
    d = ((Z[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(axis=1)

def move(Z, labels, k):
    return np.array([Z[labels == j].mean(axis=0) for j in range(k)])

def inertia_of(Z, labels, centres):
    return float(((Z - centres[labels]) ** 2).sum())

rng = np.random.default_rng(0)
centres = Z[rng.choice(len(Z), 3, replace=False)].copy()

labels = assign(Z, centres)
i1 = inertia_of(Z, labels, centres)
print("after the first assignment:", round(i1, 2))
centres = move(Z, labels, 3)
i2 = inertia_of(Z, labels, centres)
print("after moving the centres: ", round(i2, 2))
print("\nthe move lowered it by", round(i1 - i2, 2), "without touching a single label")

### Answer

The assignment gives **379.82**, and moving the centres drops it to **156.88** — a fall of
**222.95** from a step that did not relabel a single point.

That is worth pausing on. The *move* step never looks at which cluster would be better for
a point; it only recomputes three averages. It lowers inertia anyway, because the mean is
by definition the point that minimises total squared distance to a set — which is the same
fact that made least squares work on Day 33.

## Question 2

Wrap those two steps in a loop that stops when an assignment changes nobody's mind.
Print the inertia and the number of points that changed cluster at each step.

Then fit `KMeans(3, n_init=10, random_state=0)` on the same scaled data and check you got
the same inertia.

In [ ]:
def my_kmeans(Z, k, seed=0, max_iter=100):
    rng = np.random.default_rng(seed)
    centres = Z[rng.choice(len(Z), k, replace=False)].copy()
    prev = None
    for step in range(max_iter):
        labels = assign(Z, centres)
        changed = None if prev is None else int((labels != prev).sum())
        print(f"  step {step+1}:  inertia {inertia_of(Z, labels, centres):8.2f}"
              + (f"   changed: {changed}" if changed is not None else ""))
        if prev is not None and (labels == prev).all():
            print(f"  converged after {step+1} passes")
            return labels, centres
        prev = labels
        centres = move(Z, labels, k)
    return labels, centres

my_labels, my_centres = my_kmeans(Z, 3)

sk = KMeans(3, n_init=10, random_state=0).fit(Z)
mine = inertia_of(Z, my_labels, my_centres)
print()
print("mine   ", round(mine, 2))
print("sklearn", round(sk.inertia_, 2))
print("same   ", abs(mine - sk.inertia_) < 1e-6)

### Answer

Four passes, converging to an inertia of **60.61** — identical to scikit-learn's.

```
  step 1:  inertia   379.82
  step 2:  inertia   102.17   changed: 47
  step 3:  inertia    61.16   changed: 5
  step 4:  inertia    60.61   changed: 0
```

The "changed" column collapses almost immediately: 47, then 5, then 0. Nearly all the work
happens in the first pass; the rest is a handful of borderline members settling. That
pattern is typical, and it is why K-Means is so cheap compared with anything that needs
gradient descent.

Your cluster *numbers* will very likely differ from sklearn's. That is not a mistake —
cluster labels are arbitrary names, which is why we compare **inertia** and not labels.

## Question 3

Two facts about the algorithm:

- **(i)** Inertia never rises, so the loop is guaranteed to stop.
- **(ii)** The loop is **not** guaranteed to find the lowest possible inertia.

Explain why each is true, in your own words. Then say what practical step (ii) forces you
to take, and name the sklearn argument that takes it for you.

### Answer

**(i) Why inertia can never rise.** The loop only ever does two things, and neither can
make it worse:

- *Assign* sends each point to its **nearest** centre. A point either keeps the centre it
  had, or moves to one that is closer — so its own squared distance can only shrink.
- *Move* puts each centre at the **mean** of its points. The mean is precisely the
  location that minimises total squared distance to a set of points, so that step cannot
  increase the total either.

So inertia is a number that only ever falls. There are finitely many ways to split n
points into k groups, so it cannot fall forever — it must run out of room and halt. That
is the whole convergence proof.

**(ii) Why "it stops" is not "it is right".** The loop halts the instant an assignment
changes nobody's mind. That is a statement about *local* stability: every point is happy
with its own centre compared with the other two centres **as they currently stand**. It
says nothing about whether a completely different arrangement of the three centres would
be better. The algorithm cannot see such an arrangement, because it only ever takes small
downhill steps and never jumps.

This is exactly the local-minimum problem: you have walked to the bottom of *a* valley,
not *the* valley.

**What it forces you to do: run it more than once from different random starts and keep
the result with the lowest inertia.** Inertia is what makes this possible — it is a fair
score to compare runs with, since all runs are clustering the same points into the same
number of groups.

The argument is **`n_init`**, which defaults to 10 in scikit-learn. (And `init='k-means++'`,
also the default, reduces how often a start goes badly by spreading the initial centres out
deliberately rather than picking them uniformly at random.)

---
# Part B · Choosing k

## Question 4

Nobody told you there were three gym groups. Sweep `k` from 1 to 10 on the scaled gym
data and print, for each `k`, the inertia, the **drop** from the previous `k`, and the
silhouette.

Which `k` does the elbow suggest? Which does the silhouette pick?

In [ ]:
rows = []
for k in range(1, 11):
    km = KMeans(k, n_init=10, random_state=0).fit(Z)
    sil = silhouette_score(Z, km.labels_) if k > 1 else np.nan
    rows.append((k, km.inertia_, sil))

print(f"{'k':>3}{'inertia':>10}{'drop':>9}{'silhouette':>12}")
for i, (k, inert, sil) in enumerate(rows):
    drop = "" if i == 0 else f"{rows[i-1][1] - inert:9.1f}"
    print(f"{k:>3}{inert:>10.1f}{drop}{'' if k == 1 else f'{sil:12.4f}'}")

best = max(rows[1:], key=lambda r: r[2])
print(f"\nsilhouette peaks at k = {best[0]} ({best[2]:.4f})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.plot([r[0] for r in rows], [r[1] for r in rows], "-o", color="#3E7CB1", lw=2)
a1.set_xlabel("k"); a1.set_ylabel("inertia"); a1.set_title("elbow")
a2.bar([str(r[0]) for r in rows[1:]], [r[2] for r in rows[1:]],
       color=[TEAL if r[0] == best[0] else "#D9E6F2" for r in rows[1:]])
a2.set_xlabel("k"); a2.set_ylabel("silhouette"); a2.set_title("silhouette")
plt.tight_layout(); plt.show()

### Answer

```
  k   inertia     drop  silhouette
  1     520.0
  2     221.5    298.5      0.5575
  3      60.6    160.9      0.6836
  4      47.8     12.8      0.5918
  5      37.6     10.2      0.5112
  6      33.2      4.4      0.3885
```

**Both say 3, and they agree clearly.**

The elbow: the drops go 298.5, then 160.9, then **12.8**. The third cluster was worth
thirteen times more than the fourth. That cliff is unmistakable.

The silhouette: it *rises* from 0.5575 at k = 2 to **0.6836** at k = 3, then falls away.
Note that it went up — inertia can never do that, which is precisely why silhouette can
have a genuine peak and inertia cannot.

This is the comfortable case, and it is worth seeing once so you recognise it: when the
groups really are round, well separated and similar in size, every method you try points
at the same `k`.

Questions 5 and 6 show what happens when they do not.

## Question 5

Now a famous dataset with the labels stripped off. Load `load_iris()`, keep only `X`,
scale it, and sweep `k` from 2 to 6 printing the silhouette.

Which `k` does the silhouette choose?

In [ ]:
Xi, yi = load_iris(return_X_y=True)
Zi = StandardScaler().fit_transform(Xi)

for k in range(2, 7):
    km = KMeans(k, n_init=10, random_state=0).fit(Zi)
    print(f"  k={k}   inertia {km.inertia_:7.1f}   silhouette "
          f"{silhouette_score(Zi, km.labels_):.4f}")

### Answer

The silhouette picks **k = 2**, at 0.5818, well clear of k = 3 (0.4599).

## Question 6

The iris dataset has **three** real species — that is what `y` holds.

Compare `adjusted_rand_score(y, labels)` for `k = 2` and `k = 3`, and explain why the
silhouette chose the number it did. Which answer would you report?

In [ ]:
for k in [2, 3]:
    km = KMeans(k, n_init=10, random_state=0).fit(Zi)
    print(f"  k={k}   silhouette {silhouette_score(Zi, km.labels_):.4f}"
          f"   agreement with the species {adjusted_rand_score(yi, km.labels_):.4f}")

# which species get merged at k=2?
km2 = KMeans(2, n_init=10, random_state=0).fit(Zi)
print()
print(pd.crosstab(pd.Series(yi, name="true species"),
                  pd.Series(km2.labels_, name="cluster at k=2")))

### Answer

| k | silhouette | agreement with the species |
|---|---|---|
| 2 | **0.5818** | 0.5681 |
| 3 | 0.4599 | **0.6201** |

**The silhouette prefers the answer that is further from the truth.** This is the single
most useful thing to understand about clustering metrics.

The cross-tabulation shows why. Species 0 (*setosa*) sits well away from the other two,
while species 1 and 2 (*versicolor* and *virginica*) genuinely overlap — there is no clean
gap between them in these four measurements. So at k = 2 the model produces *setosa* versus
*everything else*: two tight, well-separated blobs. That is precisely the shape silhouette
is designed to reward, so it scores high.

At k = 3 the model has to cut through the overlapping region, producing two clusters that
sit close together with fuzzy borders. Silhouette punishes that — even though the result is
closer to biological reality.

**Which would I report?** It depends entirely on what the question is, and you should say
which you are answering:

- *"How many well-separated groups are in this measurement space?"* → **two**, and the
  silhouette is right.
- *"How many species are there?"* → **three**, and no clustering metric could have told
  you, because the answer lives in biology, not in the geometry of four columns.

The general lesson: a clustering score measures **separation**, not **correctness**. It
cannot know what the groups are supposed to mean. Here we could check against `y` only
because this is a teaching dataset; in a real project that column does not exist, which is
why domain knowledge is not optional.

---
# Part C · Breaking it

## Question 7

Load `load_wine()` — 178 wines, 13 chemical measurements, 3 real cultivars.

Cluster it with `k=3` **twice**: once on the raw columns, once scaled. For each, print the
**silhouette** and the **agreement with the true cultivars**.

In [ ]:
Xw, yw = load_wine(return_X_y=True)
Zw = StandardScaler().fit_transform(Xw)

for name, D in [("raw", Xw), ("scaled", Zw)]:
    km = KMeans(3, n_init=10, random_state=0).fit(D)
    print(f"  {name:7} silhouette {silhouette_score(D, km.labels_):.4f}"
          f"   agreement {adjusted_rand_score(yw, km.labels_):.4f}")

print()
print("column spreads, raw:")
print(f"  largest  {Xw.std(axis=0).max():10.2f}")
print(f"  smallest {Xw.std(axis=0).min():10.4f}")
print(f"  ratio    {Xw.std(axis=0).max()/Xw.std(axis=0).min():10.0f} : 1")

### Answer

| | silhouette | agreement with the cultivars |
|---|---|---|
| raw | **0.5711** | 0.3711 |
| scaled | 0.2849 | **0.8975** |

Scaling more than **doubles** the agreement with the real cultivars, 0.371 → 0.898.

And the reason is in the spreads: the widest column (proline) has a standard deviation
around **314**, the narrowest around **0.12** — a ratio of about **2,530 : 1**. Unscaled,
"distance between two wines" means "difference in proline", and the other twelve
measurements contribute essentially nothing.

## Question 8

Look again at the two silhouette scores you just printed. The **raw** clustering scored
**higher** than the scaled one, while being far worse.

Explain how that is possible, and say what it means for using the silhouette to make
decisions.

### Answer

**The silhouette is computed in whatever space you hand it, and it has no way of knowing
that the space is wrong.**

Unscaled, the wines effectively live on a single axis — proline — because that column
dwarfs the rest. Along one axis the points separate into three tidy, well-spaced bands,
and *within that space* those bands really are tight and far apart. Silhouette measures
exactly that, so it reports 0.5711. It is not malfunctioning; it is answering the question
it was asked, honestly, about a space we should never have used.

Scaled, all thirteen measurements get a vote. The clusters now sit in a genuine
13-dimensional space where everything is further from everything else and the boundaries
are less crisp, so the silhouette drops to 0.2849 — **even though the clusters are now
nearly the right ones**.

**What this means in practice:**

1. **Never compare silhouettes across different preprocessing.** The number is only
   meaningful between runs on *identical* features. Comparing raw against scaled is like
   comparing distances in miles against kilometres and declaring miles the winner.
2. **Use silhouette to choose `k`, not to choose your pipeline.** Within one fixed feature
   space it is a fair way to compare `k = 2` against `k = 3`. Across feature spaces it is
   meaningless.
3. **A high silhouette is not evidence that the clustering is useful.** Question 6 showed
   it preferring the wrong `k`; this shows it preferring the wrong *space*. Both times it
   was measuring separation faithfully — and separation is simply not the same thing as
   correctness.

The decision about scaling has to be made on *principle* — K-Means measures distance, so
columns must be comparable — and not by whichever option scores better.

## Question 9

Back to the gym data. Run `KMeans(k, n_init=1, init='random')` for 30 different
`random_state` values and count how many distinct inertias come back, and how many runs
got stuck in a worse one.

Do it **twice: once at `k=3` and once at `k=5`**. The difference between the two is the
point of the question.

In [ ]:
for k in [3, 5]:
    vals = {s: round(KMeans(k, n_init=1, init="random", random_state=s).fit(Z).inertia_, 2)
            for s in range(30)}
    distinct = sorted(set(vals.values()))
    best = min(distinct)
    stuck = sum(1 for v in vals.values() if v > best + 1e-9)
    print(f"k={k}:  {len(distinct):2d} distinct answers   stuck {stuck}/30   "
          f"best {best}   worst {max(distinct)}")

print()
for k in [3, 5]:
    best = min(KMeans(k, n_init=1, init="random", random_state=s).fit(Z).inertia_
               for s in range(30))
    for n in [1, 2, 5, 10]:
        hits = sum(1 for s in range(30)
                   if abs(KMeans(k, n_init=n, init="random", random_state=s).fit(Z).inertia_
                          - best) < 0.01)
        print(f"  k={k}  n_init={n:2d}  reached the best answer in {hits}/30 runs")
    print()

### Answer

```
k=3:   2 distinct answers   stuck  1/30   best  60.61   worst 209.80
k=5:  20 distinct answers   stuck 28/30   best  37.63   worst  47.60
```

**At `k = 3` the restart problem barely exists. At `k = 5` it is almost total.**

That contrast is the real lesson, and it is more useful than "always restart".

**Why k = 3 is safe here.** There are genuinely three well-separated round groups, so
almost any scattering of three centres rolls downhill into the same arrangement. There is
essentially one valley. (One seed in thirty still got stuck, at 209.80 — a very visibly
wrong answer — so even here a single run is a gamble you have no reason to take.)

**Why k = 5 is chaos.** There is no natural five-way split of this data, so there is no
single valley to roll into. The algorithm has to carve two of the real groups arbitrarily,
and *where* it carves them depends entirely on where the centres happened to land. Twenty
different answers from thirty runs.

**And `n_init` does not rescue k = 5.** Look at the second block of output:

```
k=3   n_init=1  → 29/30      k=5   n_init=1  →  2/30
k=3   n_init=2  → 30/30      k=5   n_init=2  →  2/30
k=3   n_init=5  → 30/30      k=5   n_init=5  →  5/30
k=3   n_init=10 → 30/30      k=5   n_init=10 →  6/30
```

Two restarts fixed `k = 3` completely. Ten restarts still leave `k = 5` finding its best
answer only **6 times in 30** — because with twenty near-equal valleys, ten throws of the
dice is nowhere near enough to be confident of landing in the deepest.

**So the rule is:** `n_init` is insurance against bad *luck*, not against a bad *k*.
Instability across restarts is itself a signal — if your clustering changes every time you
run it, that is usually not a nuisance to paper over, it is evidence that the `k` you asked
for does not match any real structure. Treat a wildly unstable result as a reason to
question `k`, not to restart harder.

And what makes any of this fixable is inertia: it gives a fair way to compare runs, so you
can keep the lowest. You could not do that by eye on data with more than two columns.

## Question 10

Generate `make_moons(300, noise=0.06, random_state=0)` and cluster it with `k=2`.

Print the silhouette and the agreement with the true crescents, and plot the result beside
the truth. Say in one line why K-Means cannot do this.

In [ ]:
Xm, ym = make_moons(300, noise=0.06, random_state=0)
km = KMeans(2, n_init=10, random_state=0).fit(Xm)

print("silhouette", round(silhouette_score(Xm, km.labels_), 4))
print("agreement ", round(adjusted_rand_score(ym, km.labels_), 4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show(Xm, ym, title="what is really there", ax=axes[0])
show(Xm, km.labels_, km.cluster_centers_, "what K-Means does", ax=axes[1])
plt.tight_layout(); plt.show()

### Answer

Silhouette **0.4878**, agreement **0.2343** — barely better than splitting the points at
random.

**Why it cannot work:** K-Means assigns each point to the nearest centre, so every cluster
it is capable of producing is the set of points closest to one location — a round blob.
A crescent is not a round blob, and the far tip of one crescent is physically closer to the
*other* crescent's centre than to its own.

Notice the silhouette again: **0.4878 is a middling score, not an alarm**. It is the third
time in this problem set that the metric has failed to warn us. Question 6 it chose the
wrong `k`, Question 8 the wrong space, and here it shrugs at a result that is essentially
worthless. Plot your clusters.

(Day 44 introduces DBSCAN, which handles this exact shape perfectly.)

---
# Part D · Using it

## Question 11

Cluster the gym data with `k=3` on the scaled columns, then build a profile table showing,
for each cluster: how many members, their average **visits per month**, their average
**minutes per visit**, and their average **total minutes per month**.

Then name the three groups.

In [ ]:
km = KMeans(3, n_init=10, random_state=0).fit(Z)

df = pd.DataFrame(GYM, columns=["visits", "minutes"])
df["cluster"] = km.labels_
df["total_minutes"] = df["visits"] * df["minutes"]

profile = df.groupby("cluster").agg(
    members=("visits", "size"),
    visits_per_month=("visits", "mean"),
    minutes_per_visit=("minutes", "mean"),
    total_minutes=("total_minutes", "mean")).round(1)
print(profile)

print("\nagreement with the groups the data was built from:",
      round(adjusted_rand_score(GYM_TRUTH, km.labels_), 4))

### Answer

```
         members  visits_per_month  minutes_per_visit  total_minutes
cluster
0             95              19.1               38.2          730.0
1             82               6.5               98.0          640.4
2             83               6.9               35.8          247.4
```

Agreement with the groups the data was built from: **0.9675**.

| Cluster | Reading | A sensible name |
|---|---|---|
| 0 | 19.1 visits of 38 minutes | **the regulars** — short, frequent sessions |
| 1 | 6.5 visits of 98 minutes | **the weekend warriors** — rare, long sessions |
| 2 | 6.9 visits of 36 minutes | **the lapsing members** — rare and short |

The naming step is not decoration. Until a cluster has a name, nobody can do anything
different for it, and the whole exercise has produced three arbitrary integers.

## Question 12

Look at the `total_minutes` column in your profile.

Write **one sentence** a gym manager could act on this week — something that is true of
the data, specific about which group, and names a concrete action. Then say what you would
check before trusting it.

### Answer

**A sentence that works:**

> *"Our two most engaged groups spend a comparable amount of time in the gym each month —
> about 730 minutes for the 95 regulars and 640 for the 82 weekend warriors — but they
> arrive in completely different patterns, so the regulars should get off-peak weekday
> incentives while the weekend warriors need guaranteed equipment availability at
> weekends."*

It works because it (a) states a fact from the table, (b) names *which* group, and (c)
proposes something different for each. Compare with a sentence that does none of that:
*"the clustering found three groups with a silhouette of 0.52"* — true, and completely
unactionable.

A second good one, aimed at the third cluster:

> *"The 83 lapsing members average under 7 short visits a month — about 247 minutes, a
> third of either other group's gym time — so they are the ones to contact before they
> cancel."*

**What to check before trusting any of it:**

1. **Plot the clusters.** Three questions in this set have shown the metrics failing to
   notice a bad answer. Look at the picture.
2. **Check stability.** Re-run with different `random_state` values, and on a random 80%
   of the members. If the groups move around, they are not real. (This matters more than
   any single score.)
3. **Check `k` honestly.** The elbow and the silhouette both said 3 here, which is
   reassuring — but Question 6 showed the silhouette choosing the wrong `k` on iris, so
   agreement between metrics is encouraging rather than conclusive. Look at `k = 2` and
   `k = 4` as well and decide which tells the more useful story.
4. **Show it to somebody who runs the gym.** If they do not recognise the groups, the
   clustering has found geometry rather than customers — and they will usually spot in ten
   seconds what no metric can tell you.
5. **Remember nothing here was predicted.** These are descriptions of last month, not
   forecasts. "Will contact prevent cancellation?" is a different question, and answering
   it needs a labelled outcome and an experiment — Day 30's A/B test, not this.

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 3, 8, 11 and 12 are marked mostly on the explanation — a correct number
with no reasoning gets little credit.